# Modeling And Offline Evaluation

Ноутбук фиксирует экспериментальную часть проекта как отдельный артефакт принятия решений. Задача offline-контура: по истории пользователя вернуть `top-K` товаров, которые с наибольшей вероятностью будут добавлены в корзину в будущем временном окне.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

ROOT = Path.cwd().resolve()
if not (ROOT / 'artifacts').exists():
    ROOT = ROOT.parent

with (ROOT / 'artifacts' / 'reports' / 'modeling_notebook_report.json').open(encoding='utf-8') as f:
    modeling_report = json.load(f)
with (ROOT / 'artifacts' / 'reports' / 'evaluation_summary.json').open(encoding='utf-8') as f:
    evaluation_summary = json.load(f)
with (ROOT / 'artifacts' / 'reports' / 'best_model_preview.json').open(encoding='utf-8') as f:
    best_model_preview = json.load(f)

eval_frame = pd.read_parquet(ROOT / 'artifacts' / 'data' / 'evaluation_frame.parquet')
eval_frame['target_count'] = eval_frame['target_items'].map(len)
eval_frame.shape


## 1. Постановка offline-задачи

Оценивается retrieval-задача для top-K рекомендаций. Пользовательская история берётся только из train-окна, а target составляют будущие `addtocart` и `transaction` в validation-окне после исключения уже купленных товаров.


In [ ]:
offline_setup = pd.DataFrame([
    {'component': 'target events', 'value': ', '.join(modeling_report['config']['target_events'])},
    {'component': 'purchased event for exclusion', 'value': modeling_report['config']['purchased_event']},
    {'component': 'top_k', 'value': modeling_report['config']['top_k']},
    {'component': 'validation_days', 'value': modeling_report['config']['validation_days']},
    {'component': 'evaluated_users', 'value': modeling_report['evaluation_dataset']['users']},
    {'component': 'avg_target_items', 'value': modeling_report['evaluation_dataset']['avg_target_items']},
    {'component': 'avg_history_events', 'value': modeling_report['evaluation_dataset']['avg_history_events']},
])
offline_setup


**Выводы блока**

- Evaluation оптимизирует будущий `addtocart`, а не ретроспективный просмотр.
- Покупки используются одновременно как позитивный сигнал и как источник исключений для уже приобретённых товаров.
- Такой target logic приближает offline-метрику к прикладной цели сервиса: возвращать товары с шансом на cart-action, а не просто на повторный просмотр.


## 2. Почему split строго time-based

В событийных e-commerce данных будущее должно оставаться будущим. В этом проекте последние 14 дней выделены в validation-окно, а всё обучение построено только на более ранних наблюдениях.


In [ ]:
split_summary = pd.DataFrame([
    {'window': 'train', 'start': modeling_report['split_summary']['train_start'], 'end': modeling_report['split_summary']['train_end'], 'rows': modeling_report['split_summary']['train_rows']},
    {'window': 'validation', 'start': modeling_report['split_summary']['validation_start'], 'end': modeling_report['split_summary']['validation_end'], 'rows': modeling_report['split_summary']['validation_rows']},
])

fig, ax = plt.subplots(figsize=(10, 4))
split_plot = split_summary.copy()
split_plot['rows_m'] = split_plot['rows'] / 1_000_000
sns.barplot(data=split_plot, x='window', y='rows_m', hue='window', palette='Set2', legend=False, ax=ax)
ax.set_title('Объём train и validation окон')
ax.set_xlabel('window')
ax.set_ylabel('rows, millions')
plt.tight_layout()

split_summary


**Выводы блока**

- Validation находится на правом краю временной шкалы и действительно имитирует прогноз на будущее.
- Такой split исключает leakage через повторяющиеся user-item паттерны, которые при random split попали бы и в train, и в validation.
- Последние 14 дней дают разумный компромисс между свежестью holdout и числом целевых событий.


## 3. Кого оценивает offline-контур

Eval cohort не является случайной выборкой из всех пользователей. В неё входят только пользователи, у которых есть хотя бы минимальная история и хотя бы один будущий целевой item после фильтрации купленных товаров.


In [ ]:
history_bucket = pd.DataFrame(modeling_report['history_bucket_distribution'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=history_bucket, x='history_bucket', y='share', color='#4C78A8', ax=axes[0])
axes[0].set_title('Eval users по длине истории')
axes[0].set_xlabel('history events')
axes[0].set_ylabel('share')

sns.histplot(eval_frame['target_count'], bins=10, color='#F58518', ax=axes[1])
axes[1].set_title('Сколько target items приходится на пользователя')
axes[1].set_xlabel('target items in validation')
axes[1].set_ylabel('users')
plt.tight_layout()

display(history_bucket)
eval_frame[['history_events', 'target_count']].describe()


**Выводы блока**

- Около трети eval users имеют ровно одно историческое событие, поэтому offline-победитель должен хорошо работать в сверхкороткой истории.
- Даже внутри evaluation выборка остаётся разреженной: большая часть пользователей имеет очень ограниченное число будущих target items.
- Это заранее делает repeat-interest baselines особенно конкурентоспособными.


## 4. Набор baseline-моделей

Сравнивались четыре модели, каждая из которых проверяет отдельную гипотезу о структуре данных и допустимой сложности решения.


In [ ]:
pd.DataFrame(modeling_report['model_definitions'])


**Выводы блока**

- `global_popularity` задаёт нижнюю границу качества и закрывает cold-start.
- `history_baseline` проверяет гипотезу о силе повторного интереса без тяжёлого candidate generation.
- `weighted_item2item` и `hybrid` нужны не ради сложности как таковой, а ради проверки trade-off между recall и каталоговой широтой.


## 5. Сравнение моделей по метрикам

Основной критерий выбора — `Recall@10`. Дополнительно анализируются `HitRate@10`, `NDCG@10` и `catalog coverage`, чтобы не потерять понимание trade-off между качеством попаданий и шириной выдачи.


In [ ]:
metrics = pd.DataFrame(modeling_report['metrics_table']).sort_values('recall_at_k', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.barplot(data=metrics, x='model_name', y='recall_at_k', hue='model_name', palette='Blues_r', legend=False, ax=axes[0])
axes[0].set_title('Recall@10')
axes[0].tick_params(axis='x', rotation=20)
axes[0].set_xlabel('model')
axes[0].set_ylabel('value')

sns.barplot(data=metrics, x='model_name', y='ndcg_at_k', hue='model_name', palette='Greens_r', legend=False, ax=axes[1])
axes[1].set_title('NDCG@10')
axes[1].tick_params(axis='x', rotation=20)
axes[1].set_xlabel('model')
axes[1].set_ylabel('value')

sns.barplot(data=metrics, x='model_name', y='catalog_coverage', hue='model_name', palette='Oranges_r', legend=False, ax=axes[2])
axes[2].set_title('Catalog coverage')
axes[2].tick_params(axis='x', rotation=20)
axes[2].set_xlabel('model')
axes[2].set_ylabel('unique recommended items')
plt.tight_layout()

metrics


**Выводы блока**

- `history_baseline` выигрывает по `Recall@10`, `HitRate@10` и `NDCG@10`, то есть не только чаще попадает в target, но и ранжирует попадания выше.
- `weighted_item2item` и `hybrid` заметно расширяют каталог покрытия, но платят за это ухудшением core quality.
- Выбор финальной модели основан на фактическом лидерстве по offline-метрике, а не на предпочтении более сложной архитектуры.


## 6. Почему победил `history_baseline`

Для интерпретации итоговой таблицы важно разобрать качество по сегментам истории. Если модель побеждает только на одном узком кластере пользователей, это слабый аргумент. Если она стабильно сильна на ключевых сегментах, решение выглядит осознанным.


In [ ]:
bucket_metrics = pd.DataFrame(modeling_report['history_bucket_metrics'])
bucket_plot = bucket_metrics.pivot(index='history_bucket', columns='model_name', values='recall_at_k').reset_index()
bucket_long = bucket_plot.melt(id_vars='history_bucket', var_name='model_name', value_name='recall_at_k')

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=bucket_long, x='history_bucket', y='recall_at_k', hue='model_name', marker='o', ax=ax)
ax.set_title('Recall@10 по сегментам длины истории')
ax.set_xlabel('history bucket')
ax.set_ylabel('Recall@10')
plt.tight_layout()

bucket_metrics.sort_values(['history_bucket', 'recall_at_k'], ascending=[True, False])


**Выводы блока**

- `history_baseline` особенно силён там, где у пользователя уже есть явный повторный интерес: сегменты `2-4`, `5-9` и `10-19` событий.
- На сверхкороткой истории (`1` событие) hybrid иногда догоняет по hit rate, но по суммарному recall лидерство остаётся у history-based ранжирования.
- На длинной истории item-to-item retrieval лучше расширяет кандидатов, однако это расширение не компенсирует потери по core recall.


## 7. Trade-off: recall против catalog coverage

Более сложная модель имеет смысл только если дополнительная ширина каталога окупает потери по основной метрике. Здесь этот trade-off явно измерим.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=metrics, x='catalog_coverage', y='recall_at_k', hue='model_name', s=180, ax=ax)
for row in metrics.itertuples(index=False):
    ax.text(row.catalog_coverage + 20, row.recall_at_k + 0.001, row.model_name, fontsize=10)
ax.set_title('Trade-off между Recall@10 и catalog coverage')
ax.set_xlabel('catalog coverage')
ax.set_ylabel('Recall@10')
plt.tight_layout()

metrics[['model_name', 'recall_at_k', 'catalog_coverage', 'catalog_coverage_pct']]


**Выводы блока**

- `weighted_item2item` и `hybrid` почти удваивают каталоговое покрытие относительно `history_baseline`.
- Однако выигрыш по coverage не превращается в выигрыш по `Recall@10`, а значит более широкая выдача в текущем датасете не попадает в target достаточно часто.
- Для этого проекта качество попадания в целевой cart-сигнал важнее, чем максимальное разнообразие любой ценой.


## 8. Cold-start и short-history implications

Победа `history_baseline` не отменяет operational reality: большинству пользователей не хватает плотной истории для устойчивой персонализации, а в serving store остаются только пользователи с минимум двумя уникальными item interactions.


In [ ]:
serving_profile = pd.DataFrame([
    {'segment': 'train users with >=2 unique items', 'value': modeling_report['serving_profile_eligibility']['train_users_with_2plus_items_share']},
    {'segment': 'eval users with >=2 unique items', 'value': modeling_report['serving_profile_eligibility']['eval_users_with_2plus_items_share']},
    {'segment': 'eval users with any repeated target from history', 'value': modeling_report['evaluation_dataset']['share_with_repeat_target_from_history']},
    {'segment': 'eval users with full target already seen in history', 'value': modeling_report['evaluation_dataset']['share_with_full_target_in_history']},
])

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=serving_profile, x='segment', y='value', hue='segment', palette='Purples_r', legend=False, ax=ax)
ax.set_title('Сигналы short-history и repeat-interest')
ax.set_xlabel('segment')
ax.set_ylabel('share')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()

serving_profile


**Выводы блока**

- Только около пятой части train users имеют как минимум два уникальных item interactions, поэтому fallback-логика в сервисе обязательна.
- Даже в eval cohort повторный интерес наблюдается лишь у части пользователей, что объясняет потолок качества history-only модели.
- Для полностью новых пользователей итоговый сервис всё равно должен опираться на popularity fallback и принимать историю прямо в request payload.


## 9. Качественные примеры рекомендаций

Таблица ниже показывает, что именно происходит в типовых сценариях: повторный интерес, восстановление нового кандидата через item-to-item, случай, где history выигрывает у hybrid, и сложный многотаргетный промах.


In [ ]:
samples = pd.DataFrame(modeling_report['sample_cases'])
samples


**Интерпретация примеров**

- В сценариях `repeat_short_history_hit` и `repeat_medium_history_hit` history-модель возвращает именно те товары, к которым пользователь уже проявлял интерес, и этого достаточно для попадания в будущий target.
- В кейсе `item2item_recovers_new_item` retrieval действительно помогает найти новый item, который history-baseline не мог предложить напрямую.
- В сценарии `history_beats_hybrid_on_repeat_interest` добавление соседей размывает ranking и вытесняет релевантный повторный интерес.
- `difficult_multi_target_miss` показывает реальное ограничение финальной модели: когда будущая корзина уходит в сторону от исторических item-ов, одной repeat-interest логики уже недостаточно.


## 10. Итоговый выбор модели

Финальная модель проекта — `history_baseline`.

Основания выбора:

1. Лучшая offline quality по `Recall@10`, `HitRate@10` и `NDCG@10`.
2. Простая и прозрачная логика, хорошо согласованная с обнаруженным repeat-interest поведением.
3. Устойчивый serving path: модель легко сериализуется, быстро работает и естественно сочетается с popularity fallback.
4. Более сложные item-to-item варианты остаются полезными как исследовательский benchmark и как возможное направление для следующей итерации, но на текущем датасете они не оправдывают переход по сложности.
